# 53. Curvature on the revised streams

**The question.** Think of the training loss as a landscape. The model stands at
one point on it. Curvature says how sharply the ground bends there. A flat
valley is easy to move around in; a narrow steep one is not. The plasticity
literature says a model trained on many tasks in a row drifts into sharp places,
and that this is one reason it stops being able to learn.

**Why it is worth measuring again.** Notebook 42 measured this on the retired
eight-task design and found something the paper now leads with. Read at a fixed
code that belongs to no task, curvature rose 13.7, 6.5 and 5.5 times along the
stream, which looks like damage. Read at a code the model had actually trained,
on that task's own BatchNorm statistics, it did not move at all, on an arm that
had the power to see a rise that size.

That is the reading-protocol result again. The paper currently shows it on three
rank measures, which all read the same features through the same normalisation.
A reviewer can answer that the finding is about BatchNorm statistics in a
hypernetwork and nothing more. Curvature is not a rank measure and does not read
features at all, so if it splits the same way, the problem is the measuring
point and not the measure.

**What this notebook does not do.** It measures the *mean* Hessian eigenvalue,
by stepping the generator a small distance in random directions and watching the
loss. Most of the sharpness literature reports the *largest* eigenvalue, which
needs Hessian-vector products and a power iteration. That is a different
notebook and it does not exist.

**Cost.** Nothing is trained. Ten model files at about 540 MB each, and two
forward passes per measurement. At notebook 42's measured rate of about three
seconds per measurement, 3,000 measurements is roughly two and a half hours on
an A100. The run saves after every batch of rows and can be stopped and
restarted.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules

# Importing uncle twice in one session gives two copies of the module with two
# copies of its constants, which is a confusing way to fail much later.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

if IN_COLAB:
    REPO = Path("/content/path-indicators")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/sumitasthana/PATH.git", str(REPO)],
                       check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition_10x20.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this from a checkout that has the ten-group split.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None and IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package],
                       check=True)
sys.path.insert(0, str(REPO))

import numpy as np
import torch
from tqdm.auto import tqdm
from uncle import checkpoint
from uncle.config import Config, PROBE_TASK, study_defaults
from uncle.hypernet import HyperNetwork, build_target
from uncle.plasticity_indicators import REFERENCE_SEED, fixed_image_sets, reference_code
from uncle.probe import _features_and_logits
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS
from uncle.trainer import UnCLe

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                               text=True).strip()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"

# A central difference subtracts two nearly equal losses, so the digits that
# survive are the ones that carry the answer. TF32 throws away ten of them.
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

print("Running at:", HEAD)
print("Device:", GPU_NAME)

## 2. Settings

Two arms, and the difference between them is the whole experiment.

The **reference arm** stands at the fixed code seed 20261030, which belongs to
no task and was never trained, on the clean BatchNorm template. That is exactly
where every published indicator reading in this study is taken, so this arm can
be set beside them.

The **learned arm** stands at a code the model did train, on that task's own
stored BatchNorm statistics, reading that task's own pictures. That is a place
the stream actually settled, which is the place the sharpness literature means.

Giving the learned arm the clean template instead is the mistake that cost
notebook 42 a whole run: it ran a trained network on statistics it was never
calibrated for and sent the loss to ten to the eighth.

In [ ]:
STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"

ORDER = "order_A"
SEEDS = [0, 1, 2, 3, 4]

# Only boundaries 4, 9 and 10 are kept in storage for this study. Boundary 0
# needs no file, because the untrained generator is rebuilt from the seed.
#
# The reference arm gets 0, 4 and 9. Boundary 0 is the paper's starting point
# and is free; boundary 4 is there so the two arms can be compared over the
# same interval, since the learned arm cannot reach back past it.
#
# The learned arm gets 4 and 9. Group 0 is learned first in this ordering, so
# its code exists from boundary 1 onward, but boundary 1 was not kept.
REFERENCE_POINTS = [0, 4, 9]
LEARNED_POINTS = [4, 9]

# The group each arm stands on.
PROBE_GROUP = PROBE_TASK      # group 9, held out, never learned by any stream
LEARNED_GROUP = "0"           # first in ordering A, so it is learned by boundary 4

# The task each ordering unlearns at the very end, needed to rebuild a run's
# configuration because the request list is part of what names the run. The
# unlearning happens at boundary 10, so group 0 still has its code at 9.
FORGET = {"order_A": "0", "order_B": "4", "order_C": "8"}

IMAGES_PER_SET = 256
BATCH_SIZE = 64

# How far to step, as a fraction of the generator's own length. Three sizes,
# because a central difference only measures curvature while the step is small
# enough that the surface still looks like a parabola. If the three agree, the
# step was small enough.
RADII = [0.001, 0.003, 0.01]

# Directions per radius. Notebook 42 started at five and five was too few: the
# spread between directions at the end of the stream was 17 to 46 times the
# spread at the start, wider than the change being looked for. Forty cuts the
# uncertainty by about 2.8 times and made every seed a firm answer.
REPEATS = 40

# The base for the direction seeds. Kept from notebook 42 so the directions are
# drawn the same way, though the models are different ones.
CURVATURE_SEED = 20261104

RUN_LABEL = "curvature_revised_v1"
RESUME_KEY = f"uncle/curvature/{STUDY_ID}/{ORDER}/{RUN_LABEL}.jsonl"
SAVE_EVERY = 20

ARMS = ["reference", "learned"]
POINTS = {"reference": REFERENCE_POINTS, "learned": LEARNED_POINTS}
# Which models have to be loaded: every point either arm asks for, once each.
ALL_POINTS = sorted(set(REFERENCE_POINTS) | set(LEARNED_POINTS))

DATA_ROOT = (Path("/content/data/tiny-imagenet-200") if IN_COLAB
             else REPO / "data/tiny-imagenet-200")
CACHE = Path("/content/curvature") if IN_COLAB else REPO / "outputs/curvature"
CACHE.mkdir(parents=True, exist_ok=True)

rows_by_arm = {arm: len(SEEDS) * len(POINTS[arm]) * len(RADII) * REPEATS
               for arm in ARMS}
downloads = len(SEEDS) * len([p for p in ALL_POINTS if p > 0])
for arm in ARMS:
    print(f"  {arm:<10} {rows_by_arm[arm]:>5} measurements at boundaries "
          f"{POINTS[arm]}")
print(f"{sum(rows_by_arm.values())} measurements in total, {len(SEEDS)} seeds, "
      f"ordering {ORDER}.")
print(f"{downloads} model files to download, about {downloads * 0.54:.1f} GB. "
      "Both arms read the same files, so the second arm costs no extra download.")
print("Nothing trains. The cost is the downloads and two forward passes per row.")

## 3. Connect to storage

In [ ]:
def secret(name):
    """Read a credential from Colab secrets, or the environment elsewhere."""
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    value = os.environ.get(name)
    if value is None:
        env = REPO / ".env"
        if env.exists():
            for line in env.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith(name + "="):
                    return line.partition("=")[2].strip().strip(chr(34)).strip(chr(39))
    return value


store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))

# The settings the streams were trained with, read back rather than retyped, so
# the model this notebook rebuilds has the shape its checkpoint expects.
plan = json.loads(store.download(
    f"uncle/indicators/{STUDY_ID}/job2_summary.json", CACHE).read_text(encoding="utf-8"))
CLASSES = plan["classes_per_task"]
CHANCE = float(np.log(CLASSES))
print("Streams used learning rate", plan["learning_rate"], "| epochs", plan["epochs"],
      "| classes per group", CLASSES, "| chance loss", round(CHANCE, 4))


def model_key(seed, point):
    return (f"uncle/checkpoints/{STUDY_ID}/{ORDER}/seed_{seed}/"
            f"boundary_{point}.pt")


# Fail now rather than an hour in, if a model this session needs is missing.
missing = [model_key(seed, point) for seed in SEEDS for point in ALL_POINTS
           if point > 0 and store.info(model_key(seed, point)) is None]
if missing:
    raise RuntimeError("Missing models:\n" + "\n".join(sorted(missing)))
print("Every model this session needs is in storage.")

## 4. The pictures

Each arm reads its loss on held-out pictures of its own group, 256 of them,
drawn with the same seed and the same rule. Neither batch is picked by hand;
they differ only in which group they come from.

The reference arm's 256 pictures are checked against the set the published
indicator readings use, picture for picture. If they ever stop matching, the
curvature and the indicators would be describing different images.

In [ ]:
needed = tuple([str(i) for i in range(9)] + [PROBE_TASK])
tasks = build_tasks(root=DATA_ROOT, partition_path=PARTITIONS[20],
                    partition_seed=42, include=needed, download=True)
print("Loaded Tiny ImageNet images for", len(needed), "groups")


def labelled_set(group):
    """The 256 fixed pictures of one group, with their labels kept.

    `fixed_image_sets` returns images only, because the indicators never need a
    label. Curvature needs the loss, so it needs labels too. This repeats that
    function's draw exactly: same seed, same permutation, same count.
    """
    pool = tasks[group]["test"]
    stream = torch.Generator().manual_seed(REFERENCE_SEED)
    picked = torch.randperm(len(pool), generator=stream)[:IMAGES_PER_SET]
    images = torch.stack([pool[int(i)][0] for i in picked])
    labels = torch.tensor([pool[int(i)][1] for i in picked])
    return images, labels


BATCHES = {"reference": labelled_set(PROBE_GROUP),
           "learned": labelled_set(LEARNED_GROUP)}

# The check that this is the same reading set the rest of the study uses.
same = fixed_image_sets({"probe": tasks[PROBE_GROUP]["test"]}, count=IMAGES_PER_SET)
if not torch.equal(BATCHES["reference"][0], same["probe"]):
    raise RuntimeError("The labelled batch is not the set the indicators used.")

for arm, (images, labels) in BATCHES.items():
    group = PROBE_GROUP if arm == "reference" else LEARNED_GROUP
    print(f"{arm:<10} group {group}: {tuple(images.shape)}, "
          f"{len(set(labels.tolist()))} classes present")

## 5. Rebuilding a model

Boundary 0 is not a file. It is the untrained generator, rebuilt from the seed.
The four random number generators are seeded in the same order the study seeds
them, because `HyperNetwork.__init__` reads the global random state. At a loaded
boundary the checkpoint overwrites all of it, so this only matters at 0.

In [ ]:
import gc
import random


def build(seed, point):
    """That seed's model at `point` in ordering A."""
    settings = study_defaults(ORDER)
    settings["requests"] = settings["requests"] + (("forget", FORGET[ORDER]),)
    config = Config(**settings, seed=seed,
                    learning_rate=plan["learning_rate"], epochs=plan["epochs"],
                    batch_size=plan["batch_size"],
                    initialization=plan["initialization"], device=DEVICE)
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(config.seed)
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=False)
    path = None
    if point > 0:
        path = store.download(model_key(seed, point), CACHE)
        payload = torch.load(path, map_location=DEVICE, weights_only=False)
        checkpoint.restore(payload, hypernet=hypernet, uncle=trainer)
        del payload
    return target, hypernet, trainer, path


def code_for(arm, hypernet):
    """Where this arm stands: a code belonging to nothing, or a learned one."""
    if arm == "reference":
        return reference_code(hypernet)
    if LEARNED_GROUP not in hypernet.task_codes:
        raise RuntimeError(
            f"This model has no code for group {LEARNED_GROUP}, so it never "
            "learned it. The learned arm has nothing to stand on here.")
    return hypernet.task_codes[LEARNED_GROUP].detach()


def buffers_for(arm, trainer):
    """The BatchNorm running statistics that belong with this arm's code.

    The hypernetwork writes weights, not buffers, so the trainer keeps one set
    of running mean and variance per group and updates it only while that group
    is being learned.

    The reference arm stands at a code belonging to no group, so no set is its
    own, and the clean template every group starts from is the only defensible
    choice. It is also what the indicators use, which is what keeps this arm
    comparable with them.

    The learned arm stands at a code the model trained, so it gets that group's
    own set.
    """
    if arm == "reference":
        return trainer.buffer_template
    if LEARNED_GROUP not in trainer.task_buffers:
        raise RuntimeError(
            f"This model has no BatchNorm statistics for group {LEARNED_GROUP}.")
    return trainer.task_buffers[LEARNED_GROUP]


BUFFER_NAME = {"reference": "clean template", "learned": "that group's own"}

## 6. Pick up an interrupted run

Every finished measurement is named by arm, seed, boundary, radius and
repetition. A row already in the file is skipped, so stopping the runtime and
starting again costs only what was in flight.

In [ ]:
PARTIAL = CACHE / f"{RUN_LABEL}.jsonl"
results = []
if store.info(RESUME_KEY) is not None:
    text = store.download(RESUME_KEY, CACHE).read_text(encoding="utf-8")
    results = [json.loads(line) for line in text.splitlines() if line.strip()]
    print(f"Resuming: {len(results)} measurements already saved.")
else:
    print("Starting fresh.")


def identity(row):
    return (row["arm"], row["seed"], row["point"], row["radius"],
            row["repetition"])


done = {identity(row) for row in results}
PARTIAL.write_text("".join(json.dumps(r) + "\n" for r in results),
                   encoding="utf-8")
todo = sum(1 for arm in ARMS for seed in SEEDS for point in POINTS[arm]
           for radius in RADII for rep in range(REPEATS)
           if (arm, seed, point, radius, rep) not in done)
print(f"{todo} measurements left to make.")

## 7. Measure the curvature

For one direction: step the generator a small distance that way and read the
loss, come back, step the same distance the other way and read the loss again.
Average the two and subtract the loss in the middle. Going both ways makes the
two slope terms cancel, because they are equal and opposite, so what is left is
the bend alone. Dividing by the squared step turns it into the curvature in that
direction, and averaged over random directions that is the mean eigenvalue of
the Hessian.

In [ ]:
def direction_seed(seed, point, radius, repetition):
    """A seed of its own for every draw, built from the row's identity.

    The arm is deliberately left out, so both arms see the same directions at
    the same point. They stand in different places on different pictures, so
    nothing is shared between their answers; it just removes one more source of
    difference between them.
    """
    return (CURVATURE_SEED + 100000 * seed + 1000 * point
            + 10 * RADII.index(radius) + repetition)


@torch.no_grad()
def generator_size(hypernet):
    """The length of the generator's weight vector, in double precision.

    A float32 sum over 135 million squared entries loses enough digits to shift
    the step size by more than the effect being measured.
    """
    return float(torch.sqrt(sum((p.double() ** 2).sum()
                                for p in hypernet.generator_parameters())))


@torch.no_grad()
def draw_direction(hypernet, seed):
    """One random direction over the whole generator, scaled to unit length.

    Drawn evenly over every entry, so the large tensors take almost all of the
    movement simply because they hold almost all of the entries. Task codes are
    not generator parameters, so a code is never moved: only the weights that
    write the target.
    """
    params = hypernet.generator_parameters()
    gen = torch.Generator(device=params[0].device).manual_seed(seed)
    noise = [torch.randn(p.shape, generator=gen, device=p.device, dtype=p.dtype)
             for p in params]
    length = float(torch.sqrt(sum((n.double() ** 2).sum() for n in noise)))
    for n in noise:
        n.div_(length)
    return noise


@torch.no_grad()
def step(hypernet, noise, scale):
    """Add `scale` times a stored direction to every generator parameter."""
    for p, n in zip(hypernet.generator_parameters(), noise):
        p.add_(n * scale)


@torch.no_grad()
def loss_at(hypernet, target, trainer, task_code, images_all, labels_all, buffers):
    """Mean cross-entropy on the fixed batch, for the weights right now.

    The model is forced into evaluation mode for the same reason the indicators
    do it: a BatchNorm layer left in training mode rescales using whatever batch
    is in front of it, which would hide what is being measured.
    """
    weights = hypernet.weights_from_code(task_code)
    modes = hypernet.training, target.training
    hypernet.eval()
    target.eval()
    total = 0.0
    try:
        for start in range(0, len(images_all), BATCH_SIZE):
            images = images_all[start:start + BATCH_SIZE].to(trainer.device)
            labels = labels_all[start:start + BATCH_SIZE].to(trainer.device)
            _, logits = _features_and_logits(hypernet, target, trainer, None,
                                             images, weights=weights,
                                             buffers=buffers)
            total += float(torch.nn.functional.cross_entropy(
                logits, labels, reduction="sum"))
    finally:
        hypernet.train(modes[0])
        target.train(modes[1])
    return total / len(images_all)


def save(row):
    results.append(row)
    done.add(identity(row))
    with PARTIAL.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row) + "\n")


began = time.perf_counter()
new_rows = 0

for seed in SEEDS:
    for point in ALL_POINTS:
        # Both arms that want this boundary, so the file is downloaded once.
        work = {arm: [(radius, rep) for radius in RADII for rep in range(REPEATS)
                      if (arm, seed, point, radius, rep) not in done]
                for arm in ARMS if point in POINTS[arm]}
        work = {arm: left for arm, left in work.items() if left}
        if not work:
            print(f"seed {seed}, boundary {point}: already finished")
            continue

        print(f"\nseed {seed}, boundary {point}: loading "
              f"({', '.join(work)})")
        target, hypernet, trainer, path = build(seed, point)
        try:
            size = generator_size(hypernet)
            saved = [p.detach().clone() for p in hypernet.generator_parameters()]

            for arm, left in work.items():
                task_code = code_for(arm, hypernet)
                images_all, labels_all = BATCHES[arm]
                buffers = buffers_for(arm, trainer)
                centre = loss_at(hypernet, target, trainer, task_code,
                                 images_all, labels_all, buffers)
                print(f"  {arm}: generator size {size:.2f}, loss where it "
                      f"stands {centre:.4f} on {BUFFER_NAME[arm]} statistics "
                      f"(chance is {CHANCE:.4f})")
                # The check that caught the mismatched-statistics bug on the
                # old design. A trained code on its own statistics reading its
                # own pictures cannot be fifty times chance. If it is, the
                # weights and the statistics do not belong together and the
                # bend measured here would mean nothing.
                if arm == "learned" and centre > 50 * CHANCE:
                    raise RuntimeError(
                        f"Group {LEARNED_GROUP} has loss {centre:.1f} at "
                        f"boundary {point}, where chance is {CHANCE:.3f}. The "
                        "weights and the BatchNorm statistics do not match.")

                bar = tqdm(left, desc=f"{arm} seed {seed} b={point}",
                           unit="measurement")
                for radius, rep in bar:
                    distance = radius * size
                    noise = draw_direction(
                        hypernet, direction_seed(seed, point, radius, rep))

                    step(hypernet, noise, distance)
                    plus = loss_at(hypernet, target, trainer, task_code,
                                   images_all, labels_all, buffers)
                    # Restore rather than step back twice, so float error does
                    # not pile up across the draws at this point.
                    with torch.no_grad():
                        for p, s in zip(hypernet.generator_parameters(), saved):
                            p.copy_(s)
                    step(hypernet, noise, -distance)
                    minus = loss_at(hypernet, target, trainer, task_code,
                                    images_all, labels_all, buffers)
                    with torch.no_grad():
                        for p, s in zip(hypernet.generator_parameters(), saved):
                            p.copy_(s)
                    del noise

                    rise = 0.5 * (plus + minus) - centre
                    curvature = 2.0 * rise / (distance ** 2)
                    save(dict(study=STUDY_ID, order=ORDER, arm=arm,
                              buffers=BUFFER_NAME[arm], seed=seed, point=point,
                              radius=radius, repetition=rep,
                              loss_centre=centre, loss_plus=plus,
                              loss_minus=minus, rise=rise, step_norm=distance,
                              generator_norm=size, curvature=curvature,
                              run_label=RUN_LABEL, code_revision=HEAD,
                              device=GPU_NAME))
                    new_rows += 1
                    if new_rows % SAVE_EVERY == 0:
                        store.upload(PARTIAL, RESUME_KEY, overwrite=True)
                bar.close()
        finally:
            del trainer, hypernet, target
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            # One model file at a time, so five seeds do not fill the disk.
            if path is not None and Path(path).exists():
                Path(path).unlink()
        store.upload(PARTIAL, RESUME_KEY, overwrite=True)

store.upload(PARTIAL, RESUME_KEY, overwrite=True)
print(f"\n{len(results)} measurements in total, {new_rows} in this session "
      f"({(time.perf_counter() - began) / 60:.1f} minutes).")
left = sum(1 for arm in ARMS for seed in SEEDS for point in POINTS[arm]
           for radius in RADII for rep in range(REPEATS)
           if (arm, seed, point, radius, rep) not in done)
print("Nothing left to run." if left == 0 else
      f"{left} still to do. Run this notebook again from the top.")

## 8. Read the result

Three things decide the answer.

**Does the reference arm rise?** Compare boundary 0 with boundary 9 on each
seed, against a floor built from the spread at both ends. If it rises, curvature
joins the rank measures in calling the model damaged.

**Does the learned arm stay still?** Compare boundary 4 with boundary 9. Staying
still only means something if the arm could have seen a rise, so the table also
prints what size of change that arm had the power to detect. A flat reading on
an arm that could not see anything is not evidence.

**Do the three step sizes agree?** If they point in different directions, the
measurement is not converged and nothing above can be read.

In [ ]:
import math
from statistics import mean, stdev


def rows_for(arm, seed, point):
    return [r for r in results if r["arm"] == arm and r["seed"] == seed
            and r["point"] == point]


def floor(first, second):
    """The smallest change that would survive the noise at both ends.

    The usual 95 percent interval for a difference of two means. If the change
    is smaller than this, the two readings cannot be told apart.
    """
    if len(first) < 2 or len(second) < 2:
        return float("inf")
    pooled = math.sqrt((stdev(first) ** 2 + stdev(second) ** 2) / 2)
    return 1.96 * pooled * math.sqrt(1 / len(first) + 1 / len(second))


print(f"CURVATURE ON {ORDER}, {len(SEEDS)} SEEDS, STUDY {STUDY_ID}")
print("Mean Hessian eigenvalue, averaged over "
      f"{REPEATS} directions at each of {len(RADII)} step sizes.\n")

for arm in ARMS:
    start, end = POINTS[arm][0], POINTS[arm][-1]
    print(f"{arm.upper()} ARM: boundary {start} to boundary {end}, "
          f"{BUFFER_NAME[arm]} statistics")
    print(f"{'seed':>6}{'start':>14}{'end':>14}{'ratio':>10}"
          f"{'change':>14}{'floor':>14}{'verdict':>10}")
    for seed in SEEDS:
        first = [r["curvature"] for r in rows_for(arm, seed, start)]
        second = [r["curvature"] for r in rows_for(arm, seed, end)]
        if not first or not second:
            print(f"{seed:>6}{'not measured':>14}")
            continue
        a, b = mean(first), mean(second)
        change, bar = b - a, floor(first, second)
        verdict = "moved" if abs(change) > bar else "flat"
        ratio = b / a if a else float("nan")
        print(f"{seed:>6}{a:>14.4g}{b:>14.4g}{ratio:>10.2f}"
              f"{change:>14.4g}{bar:>14.4g}{verdict:>10}")
    print()

# The loss at the measuring point, because notebook 42's reference-arm rise
# turned out to track the loss there rather than anything about the model.
print("LOSS WHERE EACH ARM STANDS (chance is "
      f"{CHANCE:.3f})")
print(f"{'arm':>12}{'seed':>6}{'boundary':>10}{'loss':>14}")
for arm in ARMS:
    for seed in SEEDS:
        for point in POINTS[arm]:
            here = rows_for(arm, seed, point)
            if here:
                print(f"{arm:>12}{seed:>6}{point:>10}"
                      f"{here[0]['loss_centre']:>14.4g}")

# If the three step sizes disagree in direction, the central difference has not
# converged and the table above cannot be read.
print("\nSTEP SIZE CHECK: the same comparison at each radius on its own")
for arm in ARMS:
    start, end = POINTS[arm][0], POINTS[arm][-1]
    for radius in RADII:
        changes = []
        for seed in SEEDS:
            first = [r["curvature"] for r in rows_for(arm, seed, start)
                     if r["radius"] == radius]
            second = [r["curvature"] for r in rows_for(arm, seed, end)
                      if r["radius"] == radius]
            if first and second:
                changes.append(mean(second) - mean(first))
        if changes:
            ups = sum(1 for c in changes if c > 0)
            print(f"  {arm:<10} radius {radius:<7g} mean change "
                  f"{mean(changes):>12.4g}  ({ups} of {len(changes)} seeds up)")